# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the FAIR^2 dataset (mass spectrometry of extracellular vesicles) using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema and is accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`. This provides information about the dataset, including available record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata and access dataset object
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets and their `@id`s. Each record set represents a logical table, often with columns and fields representing its contents. All references are made via the `@id` field.

In [ ]:
# Inspect available record sets
print("Record Sets (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"- {rs.id} (name: '{rs.name}')")

# List fields (columns) in each record set
print("\nRecord Set Field Overview (by @id):\n")
for rs in dataset.metadata.record_sets:
    print(f"Record Set: {rs.id} (name: '{rs.name}')")
    for field in rs.fields:
        print(f"  - Field @id: {field.id}, name: '{field.name}', type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. The record set and field `@id`s from the overview are used to ensure correct mapping.

In [ ]:
# Build a list of available record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Load all record sets into DataFrames, keying by @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()  # in case it's empty

# Choose a primary record set for demonstration (here, take the first one)
main_record_set_id = record_set_ids[0]
print(f"\nColumn names in chosen record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. All selection and filtering are conducted using field `@id`s as column names.

In [ ]:
df = dataframes[main_record_set_id]

# List numeric fields in the chosen record set
numeric_fields = []
for field in dataset.metadata.find_record_set(main_record_set_id).fields:
    if field.data_type in ('schema:Float', 'schema:Number', 'schema:Integer') and field.id in df.columns:
        numeric_fields.append(field.id)
print(f"Numeric fields in record set {main_record_set_id}: {numeric_fields}")

# Select a numeric field for filtering and normalization
if numeric_fields:
    numeric_field = numeric_fields[0]

    # Show a histogram and filter by the mean as a demonstration
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with '{numeric_field}' > mean ({threshold:.2f}):\n")
    print(filtered_df.head())

    # Normalize the field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if one exists
    group_fields = [field.id for field in dataset.metadata.find_record_set(main_record_set_id).fields if field.data_type == 'schema:Text' and field.id in df.columns]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric fields detected; cannot perform numeric EDA.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and group-based aggregates.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    # Histogram of raw numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot of numeric_field grouped by group_field (if available)
    if 'group_field' in locals():
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook has demonstrated how to load the FAIR^2 mass spectrometry EV dataset using `mlcroissant`, examine its structure, perform basic filtering and normalization using field and record set `@id` references, and visualize numeric data distributions.

All data operations are traceable to the Croissant schema, ensuring reproducibility and FAIR compliance for downstream data science workflows.